# Stage 17 — карта информационного разрыва текущего датасета

## Исследовательский вопрос

**Какой информации потенциально не хватает текущему набору из 47 разрешённых признаков для описания клиентов, которые остаются сложными для базовых моделей?**

---

## Почему этот вопрос возникает сейчас

Предыдущие этапы показали:

- на текущем наборе из 47 разрешённых признаков построен сильный воспроизводимый GBDT baseline;
- проверка нескольких современных модельных подходов не дала устойчивого существенного улучшения;
- Stage 16 подтвердил существование общей группы из 805 дефолтных клиентов, которых одновременно не обнаружили CatBoost, XGBoost и LightGBM;
- эта группа имеет наблюдаемый профиль признаков;
- проблема не объясняется пропусками данных;
- для KMeans при `k=2..6` silhouette не показал сильного разделения на отдельные типы; строгая устойчивость кластеров отдельно не проверялась..

Эти результаты не доказывают, что достигнут абсолютный потолок качества.

Однако они делают следующий вопрос более ценным, чем очередной поиск архитектуры модели:

**какие аспекты состояния и поведения компании вообще представлены в текущем датасете, а какие потенциально отсутствуют?**

---

## Что проверяем

В Stage 17 необходимо:

1. восстановить точный состав 47 разрешённых признаков;
2. сопоставить известные признаки с их бизнес-смыслом;
3. определить, какие информационные области уже представлены в датасете;
4. выделить области, которые текущие признаки не описывают или описывают ограниченно;
5. сформировать гипотезы дополнительных факторов;
6. отделить гипотезы, которые можно проверить на текущем датасете, от тех, для которых потребуется новый источник данных или исторический датасет.

---

## Что остаётся неизменным

- Dataset: `Data_final.xlsb`.
- Target: `DefMark`.
- Identifier: `INN`.
- Рабочее пространство модели: 47 признаков.
- `Q_B1_norm` и `Q_B2_norm` не являются predictors финальной рабочей модели.
- Final test не используется.
- Новые модели не обучаются.
- Hyperparameter tuning не выполняется.
- Порог классификации не исследуется.

---

## Критическое временное ограничение

В предоставленном датасете отсутствует надёжная row-level дата наблюдения.

Поэтому:

- random CV/OOF не доказывает временную стабильность;
- текущие внешние данные нельзя трактовать как исторические признаки старых строк;
- историческое внешнее обогащение нельзя честно выполнить без нового временного основания.

---

## Критерий завершения Stage 17

Этап считается завершённым, если получена воспроизводимая карта:

**текущая информация → наблюдаемые ограничения → гипотезы недостающей информации → возможность проверки.**

Результатом Stage 17 должен быть не новый прогноз качества модели, а обоснованное решение:

**что ещё можно исследовать на текущем датасете, а для каких вопросов уже потребуется новый исторический набор данных.**

# 1. Восстановление рабочего пространства признаков

## Исследовательский вопрос

Какие именно признаки составляют текущее рабочее информационное пространство модели?

## Зачем это проверять

Stage 17 должен анализировать не абстрактный набор факторов, а тот же самый feature contract, который использовался в принятом исследовании.

Поэтому сначала восстанавливаем точный список разрешённых признаков из сохранённого артефакта Stage 16 и проверяем:

- количество признаков;
- отсутствие `Q_B1_norm`;
- отсутствие `Q_B2_norm`.

Никакие признаки на этом этапе не добавляются и не удаляются.

In [1]:
# ============================================================
# 0.1 Контроль выполнения долгих операций
# Показывает, что kernel работает, и сколько времени прошло.
# ============================================================

import time
import threading
from contextlib import contextmanager


HEARTBEAT_SECONDS = 10


@contextmanager
def heartbeat(stage: str):
    started = time.monotonic()
    stop_event = threading.Event()

    print(f"▶ {stage}: старт", flush=True)

    def worker():
        while not stop_event.wait(HEARTBEAT_SECONDS):
            elapsed = time.monotonic() - started
            print(
                f"⏱ {stage}: выполняется | прошло {elapsed:.0f} сек.",
                flush=True
            )

    thread = threading.Thread(
        target=worker,
        daemon=True
    )
    thread.start()

    try:
        yield
    except Exception:
        elapsed = time.monotonic() - started
        print(
            f"❌ {stage}: ошибка после {elapsed:.1f} сек.",
            flush=True
        )
        raise
    else:
        elapsed = time.monotonic() - started
        print(
            f"✅ {stage}: завершено за {elapsed:.1f} сек.",
            flush=True
        )
    finally:
        stop_event.set()
        thread.join(timeout=1)

In [2]:
# ============================================================
# 1.1 Восстановление точного feature contract
# Цель: получить те же 47 разрешённых признаков,
# которые использовались в принятом Stage 16.
#
# Новые признаки не создаются.
# Модели не обучаются.
# ============================================================

import json
from pathlib import Path


STAGE16_PATH = Path(
    "../reports/generated/stage16_blind_spot_inventory_V1.json"
)


with STAGE16_PATH.open(
    "r",
    encoding="utf-8"
) as file:
    stage16 = json.load(file)


feature_names = stage16["analysis_parameters"]["features"]


print("Количество разрешённых признаков:")
print(len(feature_names))

print()

print("Q_B1_norm присутствует:")
print("Q_B1_norm" in feature_names)

print()

print("Q_B2_norm присутствует:")
print("Q_B2_norm" in feature_names)

print()

print("Список 47 признаков:")
for number, feature in enumerate(feature_names, start=1):
    print(f"{number:>2}. {feature}")

Количество разрешённых признаков:
47

Q_B1_norm присутствует:
False

Q_B2_norm присутствует:
False

Список 47 признаков:
 1. Q_A1_norm
 2. Q_A2_norm
 3. Q_A3_norm
 4. Q_A4_norm
 5. Q_A5_norm
 6. Q_A6_norm
 7. Q_A7_norm
 8. Q_B3_norm
 9. Q_B4_norm
10. Q_B5_norm
11. Q_C1_norm
12. Q_D1_norm
13. Q_D2_norm
14. Q_D3_norm
15. Q_D4_norm
16. Q_D5_norm
17. Q_D6_norm
18. A1_norm
19. A2_norm
20. A3_norm
21. A4_norm
22. A5_norm
23. A6_norm
24. B1_norm
25. B2_norm
26. B3_norm
27. C1_norm
28. C2_norm
29. C3_norm
30. C4_norm
31. D1_norm
32. D2_norm
33. D3_norm
34. D4_norm
35. D5_norm
36. E1_norm
37. E2_norm
38. E3_norm
39. F1_norm
40. F2_norm
41. F3_norm
42. F4_norm
43. G1_norm
44. G2_norm
45. G3_norm
46. G4_norm
47. G5_norm


# Результат проверки feature contract

Рабочее информационное пространство восстановлено корректно:

- 47 разрешённых признаков;
- `Q_B1_norm` отсутствует;
- `Q_B2_norm` отсутствует.

Feature contract совпадает с принятым исследовательским протоколом.

## Следующий вопрос

Какую часть этих 47 признаков мы можем содержательно интерпретировать по информации заказчика?

In [3]:
# ============================================================
# 1.2 Карта известных расшифровок признаков
# Цель: определить, какую часть из 47 признаков мы уже
# можем интерпретировать по информации заказчика.
#
# Неизвестные признаки не расшифровываются предположениями.
# ============================================================

import pandas as pd


feature_meanings = {
    # Финансовые признаки
    "A1_norm": "Собственный капитал / Активы",
    "A4_norm": "Капитал и резервы + ДБП",
    "B1_norm": "Внеоборотные активы / Активы",
    "B2_norm": "Долг / Активы",
    "D1_norm": "EBIT / Активы",
    "D5_norm": "Чистая прибыль / Выручка",
    "E1_norm": "Выручка / Дебиторская задолженность",
    "G1_norm": "Темп роста выручки",
    "G2_norm": "Темп роста чистых активов",

    # Нефинансовые / качественные признаки
    "Q_A1_norm": "Регион регистрации",
    "Q_A4_norm": "Отраслевая характеристика",
    "Q_A5_norm": "Численность сотрудников",
    "Q_A6_norm": "Размер компании",
    "Q_B3_norm": "Количество активных исполнительных производств",
    "Q_C1_norm": "Кредитная история",
    "Q_D4_norm": "Количество компаний с аналогичным руководителем",
    "Q_D6_norm": "Режим налогообложения",
}


unknown_meanings = [
    feature
    for feature in feature_names
    if feature not in feature_meanings
]


print("Расшифровка feature space")
print("-------------------------")
print(f"Всего разрешённых признаков: {len(feature_names)}")
print(f"Расшифровано заказчиком:     {len(feature_meanings)}")
print(f"Пока без расшифровки:        {len(unknown_meanings)}")
print()


known_table = pd.DataFrame(
    [
        {
            "Признак": feature,
            "Смысл": feature_meanings[feature]
        }
        for feature in feature_names
        if feature in feature_meanings
    ]
)


display(known_table)


print()
print("Признаки без подтверждённой расшифровки:")

for number, feature in enumerate(unknown_meanings, start=1):
    print(f"{number:>2}. {feature}")

Расшифровка feature space
-------------------------
Всего разрешённых признаков: 47
Расшифровано заказчиком:     17
Пока без расшифровки:        30



,Признак,Смысл
0,Q_A1_norm,Регион регистрации
1,Q_A4_norm,Отраслевая характеристика
2,Q_A5_norm,Численность сотрудников
3,Q_A6_norm,Размер компании
4,Q_B3_norm,Количество активных исполнительных производств
5,Q_C1_norm,Кредитная история
6,Q_D4_norm,Количество компаний с аналогичным руководителем
7,Q_D6_norm,Режим налогообложения
8,A1_norm,Собственный капитал / Активы
9,A4_norm,Капитал и резервы + ДБП



Признаки без подтверждённой расшифровки:
 1. Q_A2_norm
 2. Q_A3_norm
 3. Q_A7_norm
 4. Q_B4_norm
 5. Q_B5_norm
 6. Q_D1_norm
 7. Q_D2_norm
 8. Q_D3_norm
 9. Q_D5_norm
10. A2_norm
11. A3_norm
12. A5_norm
13. A6_norm
14. B3_norm
15. C1_norm
16. C2_norm
17. C3_norm
18. C4_norm
19. D2_norm
20. D3_norm
21. D4_norm
22. E2_norm
23. E3_norm
24. F1_norm
25. F2_norm
26. F3_norm
27. F4_norm
28. G3_norm
29. G4_norm
30. G5_norm


# 1.3 Полнота бизнес-расшифровки feature space

## Факты

Из 47 разрешённых признаков:

- 17 имеют подтверждённую расшифровку от заказчика;
- 30 пока не имеют подтверждённого бизнес-смысла.

Таким образом, содержательная интерпретация текущего feature space пока неполна.

## Интерпретация

На этом этапе нельзя делать сильный вывод, что определённый тип информации отсутствует в датасете.

Причина:

часть неизвестных 30 признаков может уже описывать те области, которые внешне выглядят отсутствующими.

## Следующий вопрос

Можно ли восстановить расшифровку неизвестных признаков из уже существующих материалов проекта?

In [4]:
# ============================================================
# 1.3 Поиск неизвестных признаков в документации проекта
# Цель: проверить, встречаются ли 30 неизвестных признаков
# вместе с возможными пояснениями в локальных текстовых материалах.
#
# Ничего не изменяется.
# Модели не обучаются.
# ============================================================

from pathlib import Path
import time


PROJECT_ROOT = Path("..").resolve()

SEARCH_ROOTS = [
    PROJECT_ROOT / "docs",
    PROJECT_ROOT / "reports" / "summary",
]

TEXT_EXTENSIONS = {
    ".md",
    ".txt",
}


print("▶ Поиск расшифровок: старт", flush=True)

started = time.monotonic()

text_files = []

for root in SEARCH_ROOTS:
    if root.exists():
        text_files.extend(
            path
            for path in root.rglob("*")
            if path.is_file()
            and path.suffix.lower() in TEXT_EXTENSIONS
        )


print(
    f"Найдено текстовых файлов для проверки: {len(text_files)}",
    flush=True
)


matches = {feature: [] for feature in unknown_meanings}


for index, path in enumerate(text_files, start=1):

    if index % 20 == 0:
        elapsed = time.monotonic() - started
        print(
            f"⏱ Проверено файлов: {index}/{len(text_files)} "
            f"| прошло {elapsed:.1f} сек.",
            flush=True
        )

    try:
        text = path.read_text(
            encoding="utf-8",
            errors="ignore"
        )
    except Exception:
        continue

    for feature in unknown_meanings:
        if feature in text:
            matches[feature].append(
                str(path.relative_to(PROJECT_ROOT))
            )


found = {
    feature: paths
    for feature, paths in matches.items()
    if paths
}


elapsed = time.monotonic() - started

print()
print(f"✅ Поиск завершён за {elapsed:.1f} сек.")
print()
print("Неизвестных признаков:", len(unknown_meanings))
print("Найдены в документации:", len(found))
print()


for feature, paths in found.items():
    print(feature)

    for path in paths[:5]:
        print(f"  - {path}")

    if len(paths) > 5:
        print(f"  - ... ещё {len(paths) - 5}")

    print()

▶ Поиск расшифровок: старт
Найдено текстовых файлов для проверки: 11

✅ Поиск завершён за 0.2 сек.

Неизвестных признаков: 30
Найдены в документации: 5

A5_norm
  - docs\PROJECT_CONTEXT.md
  - docs\RESEARCH_RECORD.md
  - docs\ROADMAP.md

B3_norm
  - docs\PROJECT_CONTEXT.md
  - docs\RESEARCH_RECORD.md
  - docs\ROADMAP.md

C1_norm
  - docs\PROJECT_CONTEXT.md
  - docs\ROADMAP.md

C4_norm
  - docs\PROJECT_CONTEXT.md
  - docs\ROADMAP.md

D4_norm
  - docs\PROJECT_CONTEXT.md
  - docs\RESEARCH_RECORD.md
  - docs\ROADMAP.md



# 1.4 Проверка найденных упоминаний

## Факты

Из 30 признаков без подтверждённой расшифровки только 5 встречаются
в локальной текстовой документации проекта.

Сам факт упоминания признака не означает, что его бизнес-смысл известен.

Признак мог появляться:

- в результатах предыдущих экспериментов;
- в таблицах важности;
- в описании blind spot;
- без расшифровки его экономического содержания.

## Следующий вопрос

Содержат ли найденные упоминания реальную бизнес-расшифровку признаков
или только технические ссылки на их названия?

In [5]:
# ============================================================
# 1.4 Просмотр контекста найденных упоминаний
# Цель: понять, содержат ли документы реальную расшифровку
# найденных признаков или только их технические названия.
#
# Показываем максимум 2 фрагмента на признак.
# ============================================================

print("▶ Проверка контекста: старт", flush=True)

MAX_CONTEXTS_PER_FEATURE = 2
CONTEXT_LINES = 2


for feature, paths in found.items():

    print()
    print("=" * 70)
    print(f"Признак: {feature}")
    print("=" * 70)

    shown = 0

    for relative_path in paths:

        path = PROJECT_ROOT / relative_path

        try:
            lines = path.read_text(
                encoding="utf-8",
                errors="ignore"
            ).splitlines()
        except Exception:
            continue

        for line_number, line in enumerate(lines):

            if feature not in line:
                continue

            start = max(
                0,
                line_number - CONTEXT_LINES
            )

            end = min(
                len(lines),
                line_number + CONTEXT_LINES + 1
            )

            print()
            print(f"Файл: {relative_path}")
            print(
                f"Строки: {start + 1}–{end}"
            )

            for index in range(start, end):
                marker = ">" if index == line_number else " "
                print(
                    f"{marker} {index + 1:>4}: {lines[index]}"
                )

            shown += 1

            if shown >= MAX_CONTEXTS_PER_FEATURE:
                break

        if shown >= MAX_CONTEXTS_PER_FEATURE:
            break

    if shown == 0:
        print("Контекст не найден.")


print()
print("✅ Проверка контекста завершена.")

▶ Проверка контекста: старт

Признак: A5_norm

Файл: docs\PROJECT_CONTEXT.md
Строки: 177–181
   177: - final test не использован;
   178: - global SHAP и permutation importance рассчитаны на внешних validation-подвыборках;
>  179: - consensus top-10: `Q_D6_norm`, `Q_A5_norm`, `Q_B3_norm`, `Q_C1_norm`, `G1_norm`, `Q_D4_norm`, `D5_norm`, `A4_norm`, `C4_norm`, `Q_A4_norm`;
   180: - минимальная межмодельная корреляция рангов SHAP: **0.974**;
   181: - минимальная межмодельная корреляция permutation importance: **0.899**;

Файл: docs\RESEARCH_RECORD.md
Строки: 391–395
   391: - Группа имеет отличающийся профиль относительно остальных дефолтов.
   392: - Наиболее заметные различия обнаружены по признакам:
>  393:   - `Q_A5_norm`;
   394:   - `Q_D4_norm`;
   395:   - `D5_norm`;

Признак: B3_norm

Файл: docs\PROJECT_CONTEXT.md
Строки: 177–181
   177: - final test не использован;
   178: - global SHAP и permutation importance рассчитаны на внешних validation-подвыборках;
>  179: - consensus to

# 1.5 Итог полноты расшифровки признаков

## Факты

Из 47 разрешённых признаков:

- 17 имеют подтверждённую бизнес-расшифровку;
- 30 пока не имеют подтверждённого экономического содержания.

Поиск в локальной документации обнаружил отдельные упоминания неизвестных признаков,
но найденный контекст относится преимущественно к результатам предыдущих экспериментов,
а не к расшифровке их смысла.

## Интерпретация

Текущий feature space нельзя полностью интерпретировать на бизнес-уровне.

Поэтому Stage 17 не будет утверждать, что какая-либо информационная область
полностью отсутствует только потому, что её нет среди 17 расшифрованных признаков.

Для неизвестных признаков сохраняется статус:

**бизнес-смысл не подтверждён.**

## Следующий вопрос

Какие информационные области уже точно представлены среди признаков,
смысл которых подтверждён заказчиком?

In [6]:
# ============================================================
# 1.5 Карта подтверждённых информационных областей
# Цель: сгруппировать только те признаки,
# бизнес-смысл которых подтверждён заказчиком.
#
# Неизвестные 30 признаков не интерпретируются.
# ============================================================

information_domains = {
    "Регион и отрасль": [
        "Q_A1_norm",
        "Q_A4_norm",
    ],

    "Размер и масштаб компании": [
        "Q_A5_norm",
        "Q_A6_norm",
    ],

    "Исполнительные производства": [
        "Q_B3_norm",
    ],

    "Кредитная история": [
        "Q_C1_norm",
    ],

    "Связи руководства": [
        "Q_D4_norm",
    ],

    "Организационные характеристики": [
        "Q_D6_norm",
    ],

    "Капитал и структура активов": [
        "A1_norm",
        "A4_norm",
        "B1_norm",
        "B2_norm",
    ],

    "Прибыльность": [
        "D1_norm",
        "D5_norm",
    ],

    "Оборот и дебиторская задолженность": [
        "E1_norm",
    ],

    "Динамика бизнеса": [
        "G1_norm",
        "G2_norm",
    ],
}


rows = []

for domain, features in information_domains.items():
    for feature in features:
        rows.append(
            {
                "Информационная область": domain,
                "Признак": feature,
                "Смысл": feature_meanings[feature],
            }
        )


domain_table = pd.DataFrame(rows)


print("Карта подтверждённой информации")
print("--------------------------------")
print(f"Информационных областей: {domain_table['Информационная область'].nunique()}")
print(f"Расшифрованных признаков: {len(domain_table)}")
print()

display(domain_table)

Карта подтверждённой информации
--------------------------------
Информационных областей: 10
Расшифрованных признаков: 17



,Информационная область,Признак,Смысл
0,Регион и отрасль,Q_A1_norm,Регион регистрации
1,Регион и отрасль,Q_A4_norm,Отраслевая характеристика
2,Размер и масштаб компании,Q_A5_norm,Численность сотрудников
3,Размер и масштаб компании,Q_A6_norm,Размер компании
4,Исполнительные производства,Q_B3_norm,Количество активных исполнительных производств
5,Кредитная история,Q_C1_norm,Кредитная история
6,Связи руководства,Q_D4_norm,Количество компаний с аналогичным руководителем
7,Организационные характеристики,Q_D6_norm,Режим налогообложения
8,Капитал и структура активов,A1_norm,Собственный капитал / Активы
9,Капитал и структура активов,A4_norm,Капитал и резервы + ДБП


# 1.6 Переход от расшифровки к карте информационного покрытия

## Факты

Среди 17 признаков с подтверждённым бизнес-смыслом представлены 10 информационных областей:

- регион и отрасль;
- размер компании;
- исполнительные производства;
- кредитная история;
- связи руководства;
- организационные характеристики;
- капитал и структура активов;
- прибыльность;
- оборот и дебиторская задолженность;
- динамика бизнеса.

## Интерпретация

Текущий датасет содержит не только финансовую отчётность.

В нём подтверждённо представлены:

- финансовые характеристики;
- характеристики масштаба бизнеса;
- отдельные юридические и исполнительные события;
- кредитная история;
- организационные характеристики;
- отдельные признаки динамики.

Однако бизнес-смысл 30 из 47 разрешённых признаков остаётся неизвестным.

Поэтому далее используем три статуса:

1. **подтверждено представлено** — существует известный признак;
2. **не подтверждено** — среди расшифрованных признаков область не найдена, но она может скрываться среди 30 неизвестных;
3. **структурно недоступно** — информацию невозможно корректно восстановить из текущего датасета из-за известного ограничения.

## Следующий вопрос

Какие потенциально важные для годового риска области уже представлены,
какие пока не подтверждены, а какие структурно недоступны в текущих данных?

In [7]:
# ============================================================
# 1.6 Карта информационного покрытия текущего датасета
# Цель: разделить области на:
#
# - подтверждено представлено;
# - не подтверждено;
# - структурно недоступно.
#
# "Не подтверждено" НЕ означает "точно отсутствует".
# ============================================================

coverage_rows = [
    {
        "Информационная область": "Финансовое состояние компании",
        "Статус": "Подтверждено представлено",
        "Основание": "A1, A4, B1, B2, D1, D5, E1"
    },
    {
        "Информационная область": "Размер и масштаб бизнеса",
        "Статус": "Подтверждено представлено",
        "Основание": "Q_A5, Q_A6"
    },
    {
        "Информационная область": "Регион и отрасль",
        "Статус": "Подтверждено представлено",
        "Основание": "Q_A1, Q_A4"
    },
    {
        "Информационная область": "Исполнительные производства",
        "Статус": "Подтверждено представлено",
        "Основание": "Q_B3"
    },
    {
        "Информационная область": "Кредитная история",
        "Статус": "Подтверждено представлено",
        "Основание": "Q_C1"
    },
    {
        "Информационная область": "Связи руководства",
        "Статус": "Подтверждено представлено",
        "Основание": "Q_D4"
    },
    {
        "Информационная область": "Динамика выручки и чистых активов",
        "Статус": "Подтверждено представлено",
        "Основание": "G1, G2"
    },

    {
        "Информационная область": "История заказов клиента в Комусе",
        "Статус": "Не подтверждено",
        "Основание": "Среди 17 расшифрованных признаков не обнаружена"
    },
    {
        "Информационная область": "История платежей и просрочек",
        "Статус": "Не подтверждено",
        "Основание": "Среди 17 расшифрованных признаков не обнаружена"
    },
    {
        "Информационная область": "События банкротства и ликвидации",
        "Статус": "Не подтверждено",
        "Основание": "Заказчик использует такие события при определении дефолта, но их наличие среди 47 predictors не подтверждено"
    },
    {
        "Информационная область": "Статус системообразующего предприятия",
        "Статус": "Не подтверждено",
        "Основание": "Заказчик привёл фактор как пример возможного дополнительного сигнала"
    },

    {
        "Информационная область": "Историческое состояние компании на дату решения",
        "Статус": "Структурно недоступно",
        "Основание": "Нет надёжной row-level observation date"
    },
    {
        "Информационная область": "Историческая траектория внешних факторов до дефолта",
        "Статус": "Структурно недоступно",
        "Основание": "Нельзя корректно привязать внешние snapshots к историческим строкам без временного якоря"
    },
]


coverage_table = pd.DataFrame(coverage_rows)


print("Карта информационного покрытия")
print("------------------------------")

display(coverage_table)

print()
print("Количество областей по статусам:")
print(
    coverage_table["Статус"]
    .value_counts()
    .to_string()
)

Карта информационного покрытия
------------------------------


,Информационная область,Статус,Основание
0,Финансовое состояние компании,Подтверждено представлено,"A1, A4, B1, B2, D1, D5, E1"
1,Размер и масштаб бизнеса,Подтверждено представлено,"Q_A5, Q_A6"
2,Регион и отрасль,Подтверждено представлено,"Q_A1, Q_A4"
3,Исполнительные производства,Подтверждено представлено,Q_B3
4,Кредитная история,Подтверждено представлено,Q_C1
5,Связи руководства,Подтверждено представлено,Q_D4
6,Динамика выручки и чистых активов,Подтверждено представлено,"G1, G2"
7,История заказов клиента в Комусе,Не подтверждено,Среди 17 расшифрованных признаков не обнаружена
8,История платежей и просрочек,Не подтверждено,Среди 17 расшифрованных признаков не обнаружена
9,События банкротства и ликвидации,Не подтверждено,Заказчик использует такие события при определе...



Количество областей по статусам:
Статус
Подтверждено представлено    7
Не подтверждено              4
Структурно недоступно        2


# 1.7 Связь feature space с бизнес-факторами дефолта

## Исследовательский вопрос

Насколько подтверждённо расшифрованные predictors отражают события,
которые сама продуктовая методика заказчика рассматривает как факторы дефолта?

## Почему это важно

В материалах заказчика описаны восемь факторов `D_1–D_8`.

Появление любого из этих факторов в продуктовой методике переводит
вероятность дефолта в 100%.

К ним относятся:

- банкротные события;
- принудительная ликвидация;
- отзыв лицензии;
- мошенничество в отношении Комуса;
- списание дебиторской задолженности из-за неплатёжеспособности;
- прекращение исполнительного производства из-за отсутствия имущества.

Это создаёт важный исследовательский вопрос:

**представлены ли события, используемые бизнесом для характеристики дефолта,
непосредственно среди текущих 47 predictors?**

При этом отсутствие соответствия среди 17 расшифрованных признаков
не означает доказанного отсутствия во всём feature space,
поскольку бизнес-смысл ещё 30 признаков неизвестен.

In [8]:
# ============================================================
# 1.7 Карта факторов дефолта заказчика и текущих predictors
#
# Цель:
# проверить наличие прямого соответствия среди 17 признаков
# с подтверждённым бизнес-смыслом.
#
# ВАЖНО:
# "не подтверждено" != "отсутствует среди всех 47".
# ============================================================

default_factor_rows = [
    {
        "Код": "D_1",
        "Фактор дефолта заказчика":
            "Подача заявления в суд о банкротстве",
        "Связанный известный predictor": None,
        "Статус":
            "Прямое соответствие не подтверждено",
    },
    {
        "Код": "D_2",
        "Фактор дефолта заказчика":
            "Признание должника банкротом по решению суда",
        "Связанный известный predictor": None,
        "Статус":
            "Прямое соответствие не подтверждено",
    },
    {
        "Код": "D_3",
        "Фактор дефолта заказчика":
            "Введение процедур банкротства",
        "Связанный известный predictor": None,
        "Статус":
            "Прямое соответствие не подтверждено",
    },
    {
        "Код": "D_4",
        "Фактор дефолта заказчика":
            "Принудительная ликвидация",
        "Связанный известный predictor": None,
        "Статус":
            "Прямое соответствие не подтверждено",
    },
    {
        "Код": "D_5",
        "Фактор дефолта заказчика":
            "Отзыв лицензии на осуществление деятельности",
        "Связанный известный predictor": None,
        "Статус":
            "Прямое соответствие не подтверждено",
    },
    {
        "Код": "D_6",
        "Фактор дефолта заказчика":
            "Мошенничество контрагента в отношении Комуса",
        "Связанный известный predictor": None,
        "Статус":
            "Прямое соответствие не подтверждено; внутренний фактор Комуса",
    },
    {
        "Код": "D_7",
        "Фактор дефолта заказчика":
            "Списание ДЗ из-за неплатёжеспособности или невозможности взыскания",
        "Связанный известный predictor": None,
        "Статус":
            "Прямое соответствие не подтверждено; внутренний фактор Комуса",
    },
    {
        "Код": "D_8",
        "Фактор дефолта заказчика":
            "Прекращение исполнительного производства из-за отсутствия имущества",
        "Связанный известный predictor":
            "Q_B3_norm — количество активных исполнительных производств",
        "Статус":
            "Связанная область представлена, но это не прямой эквивалент D_8",
    },
]


default_factor_table = pd.DataFrame(default_factor_rows)


print("Факторы дефолта заказчика и известные predictors")
print("------------------------------------------------")

display(default_factor_table)

print()
print("Всего факторов D_1–D_8:", len(default_factor_table))

direct_matches = (
    default_factor_table["Статус"]
    == "Прямое соответствие подтверждено"
).sum()

related_only = (
    default_factor_table["Статус"]
    .str.startswith("Связанная область")
).sum()

not_confirmed = len(default_factor_table) - direct_matches - related_only

print("Прямое соответствие подтверждено:", direct_matches)
print("Только связанная область:", related_only)
print("Прямое соответствие не подтверждено:", not_confirmed)

Факторы дефолта заказчика и известные predictors
------------------------------------------------


,Код,Фактор дефолта заказчика,Связанный известный predictor,Статус
0,D_1,Подача заявления в суд о банкротстве,NaN,Прямое соответствие не подтверждено
1,D_2,Признание должника банкротом по решению суда,NaN,Прямое соответствие не подтверждено
2,D_3,Введение процедур банкротства,NaN,Прямое соответствие не подтверждено
3,D_4,Принудительная ликвидация,NaN,Прямое соответствие не подтверждено
4,D_5,Отзыв лицензии на осуществление деятельности,NaN,Прямое соответствие не подтверждено
5,D_6,Мошенничество контрагента в отношении Комуса,NaN,Прямое соответствие не подтверждено; внутренни...
6,D_7,Списание ДЗ из-за неплатёжеспособности или нев...,NaN,Прямое соответствие не подтверждено; внутренни...
7,D_8,Прекращение исполнительного производства из-за...,Q_B3_norm — количество активных исполнительных...,"Связанная область представлена, но это не прям..."



Всего факторов D_1–D_8: 8
Прямое соответствие подтверждено: 0
Только связанная область: 1
Прямое соответствие не подтверждено: 7


# 1.8 Факторы дефолта ≠ автоматически новые predictors

## Факты

Среди 17 признаков с подтверждённым бизнес-смыслом:

- прямого эквивалента факторов `D_1–D_8` не обнаружено;
- для `D_8` присутствует только связанный сигнал `Q_B3_norm`
  — количество активных исполнительных производств;
- `Q_B3_norm` не является эквивалентом события прекращения исполнительного
  производства из-за отсутствия имущества.

## Критическое различие

Фактор, используемый бизнесом для определения или подтверждения дефолта,
не становится автоматически допустимым predictor.

Для модели годовой вероятности дефолта признак допустим только тогда,
когда известно, что он был доступен **на момент принятия решения**,
то есть до прогнозируемого события.

Например:

- заявление о банкротстве;
- решение суда о банкротстве;
- ликвидация;
- списание дебиторской задолженности

могут быть очень сильными индикаторами уже наступившего или практически
наступившего дефолта.

Использование таких событий без временной привязки может привести
к target leakage.

## Ограничение текущего датасета

Надёжной row-level observation date в текущих данных нет.

Поэтому мы не можем установить для каждой исторической строки:

**событие было известно до момента прогноза или появилось после него.**

## Следующий вопрос

Какие из потенциальных новых информационных источников можно считать
допустимыми кандидатами для будущего скоринга, а какие требуют
обязательной исторической временной привязки?

In [9]:
# ============================================================
# 1.8 Temporal admissibility gate
#
# Цель:
# отделить перспективные информационные источники от факторов,
# которые нельзя честно добавить в текущий historical dataset
# без даты наблюдения и исторического состояния.
#
# Модели не обучаются.
# Final test не используется.
# ============================================================

temporal_gate_rows = [
    {
        "Источник / фактор": "Финансовая отчётность",
        "Потенциальная ценность": "Высокая",
        "Нужна историческая дата": "Да",
        "Можно добавить в текущие строки сейчас": "Нет",
        "Причина":
            "Нужно знать, какая отчётность была доступна на момент решения",
    },
    {
        "Источник / фактор": "История заказов в Комусе",
        "Потенциальная ценность": "Неизвестна — гипотеза",
        "Нужна историческая дата": "Да",
        "Можно добавить в текущие строки сейчас": "Нет",
        "Причина":
            "Нужно использовать только заказы, совершённые до момента решения",
    },
    {
        "Источник / фактор": "История платежей и просрочек перед Комусом",
        "Потенциальная ценность": "Неизвестна — гипотеза",
        "Нужна историческая дата": "Да",
        "Можно добавить в текущие строки сейчас": "Нет",
        "Причина":
            "Будущие просрочки относительно момента решения создадут leakage",
    },
    {
        "Источник / фактор": "Исполнительные производства",
        "Потенциальная ценность": "Подтверждённо связанная область",
        "Нужна историческая дата": "Да",
        "Можно добавить в текущие строки сейчас": "Нет",
        "Причина":
            "Нужно историческое состояние производств на дату решения",
    },
    {
        "Источник / фактор": "Банкротные и судебные события",
        "Потенциальная ценность": "Высокий потенциальный сигнал",
        "Нужна историческая дата": "Да — критично",
        "Можно добавить в текущие строки сейчас": "Нет",
        "Причина":
            "Событие может совпадать с target или наступать после момента прогноза",
    },
    {
        "Источник / фактор": "Ликвидация / исключение из ЕГРЮЛ",
        "Потенциальная ценность": "Высокий потенциальный сигнал",
        "Нужна историческая дата": "Да — критично",
        "Можно добавить в текущие строки сейчас": "Нет",
        "Причина":
            "Высокий риск прямого target leakage",
    },
    {
        "Источник / фактор": "Мошенничество в отношении Комуса",
        "Потенциальная ценность": "Бизнес-фактор дефолта",
        "Нужна историческая дата": "Да — критично",
        "Можно добавить в текущие строки сейчас": "Нет",
        "Причина":
            "Внутреннее событие; необходимо доказать его доступность до решения",
    },
    {
        "Источник / фактор": "Списание дебиторской задолженности",
        "Потенциальная ценность": "Бизнес-фактор дефолта",
        "Нужна историческая дата": "Да — критично",
        "Можно добавить в текущие строки сейчас": "Нет",
        "Причина":
            "Может быть следствием уже наступившей неплатёжеспособности",
    },
    {
        "Источник / фактор": "Изменения руководства / связанных компаний",
        "Потенциальная ценность": "Неизвестна — гипотеза",
        "Нужна историческая дата": "Да",
        "Можно добавить в текущие строки сейчас": "Нет",
        "Причина":
            "Нужна структура связей именно на дату решения",
    },
    {
        "Источник / фактор": "Региональный и отраслевой контекст",
        "Потенциальная ценность": "Неизвестна — гипотеза",
        "Нужна историческая дата": "Да",
        "Можно добавить в текущие строки сейчас": "Нет",
        "Причина":
            "Макро- и отраслевые показатели должны соответствовать историческому моменту",
    },
]


temporal_gate = pd.DataFrame(temporal_gate_rows)


print("Temporal admissibility gate")
print("---------------------------")

display(temporal_gate)

print()
print("Всего рассмотрено источников:", len(temporal_gate))

blocked_now = (
    temporal_gate["Можно добавить в текущие строки сейчас"] == "Нет"
).sum()

print(
    "Нельзя корректно добавить в текущие historical rows сейчас:",
    blocked_now
)

Temporal admissibility gate
---------------------------


,Источник / фактор,Потенциальная ценность,Нужна историческая дата,Можно добавить в текущие строки сейчас,Причина
0,Финансовая отчётность,Высокая,Да,Нет,"Нужно знать, какая отчётность была доступна на..."
1,История заказов в Комусе,Неизвестна — гипотеза,Да,Нет,"Нужно использовать только заказы, совершённые ..."
2,История платежей и просрочек перед Комусом,Неизвестна — гипотеза,Да,Нет,Будущие просрочки относительно момента решения...
3,Исполнительные производства,Подтверждённо связанная область,Да,Нет,Нужно историческое состояние производств на да...
4,Банкротные и судебные события,Высокий потенциальный сигнал,Да — критично,Нет,Событие может совпадать с target или наступать...
5,Ликвидация / исключение из ЕГРЮЛ,Высокий потенциальный сигнал,Да — критично,Нет,Высокий риск прямого target leakage
6,Мошенничество в отношении Комуса,Бизнес-фактор дефолта,Да — критично,Нет,Внутреннее событие; необходимо доказать его до...
7,Списание дебиторской задолженности,Бизнес-фактор дефолта,Да — критично,Нет,Может быть следствием уже наступившей неплатёж...
8,Изменения руководства / связанных компаний,Неизвестна — гипотеза,Да,Нет,Нужна структура связей именно на дату решения
9,Региональный и отраслевой контекст,Неизвестна — гипотеза,Да,Нет,Макро- и отраслевые показатели должны соответс...



Всего рассмотрено источников: 10
Нельзя корректно добавить в текущие historical rows сейчас: 10


# 1.9 Результат temporal admissibility gate

## Факты

Для всех 10 рассмотренных дополнительных информационных источников
требуется временная привязка.

Ни один из них нельзя корректно добавить к текущим историческим строкам
только по `INN`, используя современное состояние источника.

Причины различаются:

- финансовая отчётность должна соответствовать информации,
  доступной на момент решения;
- заказы, платежи и просрочки должны быть ограничены событиями,
  произошедшими до момента прогноза;
- судебные, банкротные и ликвидационные события имеют высокий риск
  target leakage;
- внутренние события Комуса также требуют доказанной даты возникновения;
- связи компаний, отраслевой и региональный контекст должны
  соответствовать историческому состоянию.

## Интерпретация

Главное ограничение следующего этапа исследования связано уже не
с отсутствием идей новых признаков.

Кандидаты на дополнительную информацию существуют.

Проблема заключается в том, что текущий датасет не содержит надёжного
временного якоря, позволяющего проверить эти гипотезы без риска
использования будущей информации.

Таким образом, простое присоединение современных данных Spark,
ФНС или других источников по `INN` не является корректным
историческим экспериментом.

## Ограничения

Этот результат не доказывает:

- что каждый из 10 источников улучшит качество модели;
- что текущие 47 признаков исчерпали всю доступную информацию;
- что среди 30 нерасшифрованных признаков нет близких сигналов;
- что новый датасет автоматически даст более высокий Gini или Recall.

Он определяет только условия,
при которых новые информационные гипотезы можно будет проверить честно.

## Следующий вопрос

Какой минимальный временной контракт должен иметь новый датасет,
чтобы проверка дополнительных факторов стала воспроизводимой
и не содержала leakage?

In [10]:
# ============================================================
# 1.9 Минимальный временной контракт будущего датасета
#
# Цель:
# определить минимальные поля и правила,
# необходимые для честного one-year PD experiment.
#
# Это дизайн данных, а не обучение модели.
# ============================================================

future_dataset_contract_rows = [
    {
        "Поле / требование": "INN",
        "Роль": "Идентификатор клиента",
        "Зачем необходимо":
            "Связь внутренних и внешних источников без использования как predictor",
        "Обязательность": "Обязательно",
    },
    {
        "Поле / требование": "observation_date",
        "Роль": "Дата состояния клиента",
        "Зачем необходимо":
            "Определяет момент, на который формируются predictors",
        "Обязательность": "Критически обязательно",
    },
    {
        "Поле / требование": "decision_date",
        "Роль": "Дата скорингового решения",
        "Зачем необходимо":
            "Фиксирует реальный момент доступности информации",
        "Обязательность": "Критически обязательно",
    },
    {
        "Поле / требование": "prediction_horizon",
        "Роль": "Горизонт прогноза",
        "Зачем необходимо":
            "Фиксирует целевой период, например 12 месяцев",
        "Обязательность": "Критически обязательно",
    },
    {
        "Поле / требование": "default_date",
        "Роль": "Дата наступления дефолта",
        "Зачем необходимо":
            "Позволяет определить, попал ли дефолт внутрь прогнозного горизонта",
        "Обязательность": "Критически обязательно для DefMark=1",
    },
    {
        "Поле / требование": "default_reason",
        "Роль": "Причина / тип дефолта",
        "Зачем необходимо":
            "Позволяет различать D_1–D_8 и внутренние бизнес-события",
        "Обязательность": "Желательно",
    },
    {
        "Поле / требование": "feature_available_at",
        "Роль": "Дата доступности каждого временного сигнала",
        "Зачем необходимо":
            "Не допускает использование информации, появившейся после решения",
        "Обязательность": "Обязательно для событийных источников",
    },
    {
        "Поле / требование": "historical_financials",
        "Роль": "Историческая финансовая отчётность",
        "Зачем необходимо":
            "Позволяет строить признаки только из данных, доступных на observation_date",
        "Обязательность": "Желательно",
    },
    {
        "Поле / требование": "internal_payment_history",
        "Роль": "История платежей / просрочек Комусу",
        "Зачем необходимо":
            "Позволяет проверить внутренние поведенческие гипотезы",
        "Обязательность": "Желательно",
    },
    {
        "Поле / требование": "internal_order_history",
        "Роль": "История заказов клиента",
        "Зачем необходимо":
            "Позволяет исследовать динамику отношений клиента с Комусом",
        "Обязательность": "Желательно",
    },
    {
        "Поле / требование": "historical_external_events",
        "Роль": "Исторические Spark / ФНС / суды / ЕГРЮЛ",
        "Зачем необходимо":
            "Позволяет проверять внешние события в состоянии до decision_date",
        "Обязательность": "Желательно",
    },
]


future_dataset_contract = pd.DataFrame(
    future_dataset_contract_rows
)


print("Минимальный контракт будущего временного датасета")
print("------------------------------------------------")

display(future_dataset_contract)

print()
print(
    "Всего требований:",
    len(future_dataset_contract)
)

print()
print("По обязательности:")
print(
    future_dataset_contract["Обязательность"]
    .value_counts()
    .to_string()
)

Минимальный контракт будущего временного датасета
------------------------------------------------


,Поле / требование,Роль,Зачем необходимо,Обязательность
0,INN,Идентификатор клиента,Связь внутренних и внешних источников без испо...,Обязательно
1,observation_date,Дата состояния клиента,"Определяет момент, на который формируются pred...",Критически обязательно
2,decision_date,Дата скорингового решения,Фиксирует реальный момент доступности информации,Критически обязательно
3,prediction_horizon,Горизонт прогноза,"Фиксирует целевой период, например 12 месяцев",Критически обязательно
4,default_date,Дата наступления дефолта,"Позволяет определить, попал ли дефолт внутрь п...",Критически обязательно для DefMark=1
5,default_reason,Причина / тип дефолта,Позволяет различать D_1–D_8 и внутренние бизне...,Желательно
6,feature_available_at,Дата доступности каждого временного сигнала,"Не допускает использование информации, появивш...",Обязательно для событийных источников
7,historical_financials,Историческая финансовая отчётность,"Позволяет строить признаки только из данных, д...",Желательно
8,internal_payment_history,История платежей / просрочек Комусу,Позволяет проверить внутренние поведенческие г...,Желательно
9,internal_order_history,История заказов клиента,Позволяет исследовать динамику отношений клиен...,Желательно



Всего требований: 11

По обязательности:
Обязательность
Желательно                               5
Критически обязательно                   3
Обязательно                              1
Критически обязательно для DefMark=1     1
Обязательно для событийных источников    1


# 1.10 Кто может предоставить необходимые данные

## Исследовательский вопрос

Какие элементы будущего временного датасета должны поступить от заказчика,
а какие потенциально можно восстановить из внешних исторических источников?

## Почему это важно

Сам факт существования потенциально полезного признака ещё не означает,
что его можно получить для исследования.

Для следующего корректного этапа необходимо разделить данные на три группы:

1. **внутренние данные Комуса** — извне их восстановить нельзя;
2. **внешние исторические данные** — потенциально можно собрать,
   но только после появления временного якоря;
3. **производные поля** — формируются после того,
   как определён корректный момент наблюдения и горизонт прогноза.

Такое разделение позволит понять,
какой минимальный запрос к заказчику необходим для продолжения исследования.

In [11]:
# ============================================================
# 1.10 Карта происхождения данных будущего датасета
#
# Цель:
# отделить данные, которые должен предоставить Комус,
# от потенциально реконструируемых внешних данных.
#
# "Потенциально реконструируемо" не означает,
# что источник уже найден или проверен.
# ============================================================

data_source_rows = [
    {
        "Элемент данных": "INN",
        "Источник": "Комус / исходная выборка",
        "Категория": "Базовый идентификатор",
        "Можно получить без заказчика": "Нет для исторической выборки",
        "Комментарий":
            "Необходим список компаний, входящих в конкретные исторические наблюдения",
    },
    {
        "Элемент данных": "decision_date",
        "Источник": "Комус",
        "Категория": "Внутренний временной якорь",
        "Можно получить без заказчика": "Нет",
        "Комментарий":
            "Это момент реального кредитного/скорингового решения",
    },
    {
        "Элемент данных": "observation_date",
        "Источник": "Комус или формируется из decision_date по зафиксированному правилу",
        "Категория": "Временной якорь",
        "Можно получить без заказчика": "Нет на текущих данных",
        "Комментарий":
            "Должно быть явно определено состояние predictors на этот момент",
    },
    {
        "Элемент данных": "prediction_horizon",
        "Источник": "Исследовательский / бизнес-протокол",
        "Категория": "Производное правило",
        "Можно получить без заказчика": "Да, после согласования определения target",
        "Комментарий":
            "Например, фиксированный горизонт 12 месяцев",
    },
    {
        "Элемент данных": "default_date",
        "Источник": "Комус + при необходимости внешняя верификация",
        "Категория": "Target data",
        "Можно получить без заказчика": "Не гарантировано",
        "Комментарий":
            "Нужно связать дату дефолта именно с принятой бизнес-дефиницией",
    },
    {
        "Элемент данных": "default_reason",
        "Источник": "Комус",
        "Категория": "Target semantics",
        "Можно получить без заказчика": "Частично",
        "Комментарий":
            "D_6 и D_7 являются внутренними событиями Комуса",
    },
    {
        "Элемент данных": "История заказов",
        "Источник": "Комус",
        "Категория": "Внутренние поведенческие данные",
        "Можно получить без заказчика": "Нет",
        "Комментарий":
            "Внешние источники не содержат историю отношений клиента с Комусом",
    },
    {
        "Элемент данных": "История платежей и просрочек",
        "Источник": "Комус",
        "Категория": "Внутренние поведенческие данные",
        "Можно получить без заказчика": "Нет",
        "Комментарий":
            "Нужны операции только до соответствующей decision_date",
    },
    {
        "Элемент данных": "Историческая финансовая отчётность",
        "Источник": "Внешние исторические источники",
        "Категория": "Потенциально реконструируемые данные",
        "Можно получить без заказчика": "Потенциально да",
        "Комментарий":
            "Требуется отдельно подтвердить источник, глубину истории и дату публикации",
    },
    {
        "Элемент данных": "Суды / банкротства / ликвидация",
        "Источник": "Внешние исторические источники",
        "Категория": "Потенциально реконструируемые события",
        "Можно получить без заказчика": "Потенциально да",
        "Комментарий":
            "Можно использовать только события, доступные до decision_date",
    },
    {
        "Элемент данных": "Исполнительные производства",
        "Источник": "Внешние исторические источники",
        "Категория": "Потенциально реконструируемые события",
        "Можно получить без заказчика": "Потенциально да",
        "Комментарий":
            "Необходимо историческое состояние, а не современный snapshot",
    },
    {
        "Элемент данных": "История руководителей / связанных компаний",
        "Источник": "Внешние исторические источники",
        "Категория": "Потенциально реконструируемые данные",
        "Можно получить без заказчика": "Потенциально да",
        "Комментарий":
            "Нужно восстановление структуры именно на дату решения",
    },
]


data_source_map = pd.DataFrame(data_source_rows)


print("Карта происхождения данных")
print("--------------------------")

display(data_source_map)

print()
print("Количество элементов по категориям:")
print(
    data_source_map["Категория"]
    .value_counts()
    .to_string()
)

Карта происхождения данных
--------------------------


,Элемент данных,Источник,Категория,Можно получить без заказчика,Комментарий
0,INN,Комус / исходная выборка,Базовый идентификатор,Нет для исторической выборки,"Необходим список компаний, входящих в конкретн..."
1,decision_date,Комус,Внутренний временной якорь,Нет,Это момент реального кредитного/скорингового р...
2,observation_date,Комус или формируется из decision_date по зафи...,Временной якорь,Нет на текущих данных,Должно быть явно определено состояние predicto...
3,prediction_horizon,Исследовательский / бизнес-протокол,Производное правило,"Да, после согласования определения target","Например, фиксированный горизонт 12 месяцев"
4,default_date,Комус + при необходимости внешняя верификация,Target data,Не гарантировано,Нужно связать дату дефолта именно с принятой б...
5,default_reason,Комус,Target semantics,Частично,D_6 и D_7 являются внутренними событиями Комуса
6,История заказов,Комус,Внутренние поведенческие данные,Нет,Внешние источники не содержат историю отношени...
7,История платежей и просрочек,Комус,Внутренние поведенческие данные,Нет,Нужны операции только до соответствующей decis...
8,Историческая финансовая отчётность,Внешние исторические источники,Потенциально реконструируемые данные,Потенциально да,"Требуется отдельно подтвердить источник, глуби..."
9,Суды / банкротства / ликвидация,Внешние исторические источники,Потенциально реконструируемые события,Потенциально да,"Можно использовать только события, доступные д..."



Количество элементов по категориям:
Категория
Внутренние поведенческие данные          2
Потенциально реконструируемые данные     2
Потенциально реконструируемые события    2
Базовый идентификатор                    1
Внутренний временной якорь               1
Временной якорь                          1
Производное правило                      1
Target data                              1
Target semantics                         1


# 1.11 Статус временных блокеров

## Исследовательский вопрос

Можно ли снять временные ограничения текущего датасета
за счёт дополнительной информации от заказчика?

## Факты

По информации заказчика:

- `DefMark` формировался на историческом периоде 2005–2024;
- надёжная row-level дата наблюдения или дата скорингового решения
  для текущих строк отсутствует;
- восстановить такой временной якорь для текущего датасета невозможно;
- точную временную схему формирования `DefMark`
  относительно каждой исторической строки восстановить также невозможно.

## Интерпретация

Оба ключевых временных условия,
необходимых для корректного исторического enrichment,
являются не открытыми вопросами, а подтверждёнными ограничениями данных.

Следовательно, дальнейшее ожидание дополнительной информации
по этим двум пунктам не требуется.

## Следующий вопрос

Как зафиксировать итоговый статус текущего датасета
и перейти к следующему объекту исследования?

In [ ]:
# ============================================================
# 1.11 Фиксация подтверждённых временных ограничений
#
# Цель:
# записать фактический статус двух критических условий.
#
# Это не запрос к заказчику:
# заказчик уже подтвердил, что восстановить их невозможно.
# ============================================================

confirmed_temporal_limits_rows = [
    {
        "Условие":
            "Row-level observation_date / decision_date",
        "Статус":
            "Недоступно и не будет восстановлено",
        "Следствие":
            "Нельзя определить исторический момент доступности predictors",
    },
    {
        "Условие":
            "Временное правило формирования DefMark относительно каждой строки",
        "Статус":
            "Недоступно и не будет восстановлено",
        "Следствие":
            "Нельзя воспроизвести строгую one-year PD temporal постановку",
    },
]


confirmed_temporal_limits = pd.DataFrame(
    confirmed_temporal_limits_rows
)


print("Подтверждённые временные ограничения")
print("------------------------------------")

display(confirmed_temporal_limits)

print()
print(
    "Критических ограничений:",
    len(confirmed_temporal_limits)
)

print(
    "Статус historical enrichment:",
    "ЗАБЛОКИРОВАН"
)

# 1.12 Итог Stage 17 — карта информационного разрыва

## Факты

В рамках Stage 17 установлено следующее.

1. Рабочий feature contract содержит **47 разрешённых признаков**.
   `Q_B1_norm` и `Q_B2_norm` в него не входят.

2. Подтверждённый бизнес-смысл имеется только для **17 из 47 признаков**.
   Смысл ещё **30 признаков** не подтверждён имеющимися материалами.

3. Среди 17 расшифрованных признаков подтверждённо представлены
   финансовые, организационные, отраслевые, кредитные и отдельные
   юридические характеристики компании.

4. Среди расшифрованных predictors не обнаружено прямого эквивалента
   бизнес-факторов дефолта `D_1–D_8`.

5. Для `D_8` присутствует связанный сигнал `Q_B3_norm`
   — количество активных исполнительных производств,
   однако он не является прямым эквивалентом события `D_8`.

6. Рассмотрены 10 потенциальных дополнительных информационных источников.
   Для всех 10 требуется историческая временная привязка.

7. В текущем датасете отсутствует надёжная row-level дата наблюдения
   или дата принятия решения.

8. Заказчик подтвердил, что row-level temporal anchor
   и точная temporal-схема формирования `DefMark`
   относительно каждой исторической строки восстановлены не будут.

---

## Интерпретация

Stage 17 не подтверждает гипотезу о том,
что текущие 47 признаков доказанно содержат недостаточно информации.

Полученный результат точнее:

**существуют обоснованные гипотезы дополнительных информационных факторов,
но их predictive value нельзя корректно проверить на текущем историческом
датасете из-за отсутствия восстанавливаемой временной структуры.**

Следовательно, ограничение следующего этапа исследования связано
не с отсутствием новых моделей и не с отсутствием идей признаков,
а с невозможностью провести временно корректный enrichment experiment.

---

## Ограничения

Stage 17 не доказывает:

- что какой-либо предложенный новый фактор улучшит Gini, Recall или PR-AUC;
- что 47 признаков исчерпали всю доступную информацию;
- что среди 30 нерасшифрованных признаков отсутствуют аналогичные сигналы;
- что группа blind spot вызвана именно недостатком информации;
- причинную природу наблюдений Stage 16;
- temporal stability существующего baseline.

Новые модели не обучались.

Final test не использовался.

---

## Решение

Историческое внешнее enrichment текущего `Data_final.xlsb`
считается заблокированным по методологическим причинам.

Современные snapshots Spark, ФНС, судебных систем, ЕГРЮЛ
и других источников нельзя присоединять к историческим строкам
только по `INN` и интерпретировать как historical predictors.

Текущий датасет сохраняется как воспроизводимый baseline
для завершённого исследования моделей и blind spot.

Следующий новый data-research объект должен иметь
корректную row-level временную структуру.

## Статус

`CURRENT_DATASET_TEMPORAL_ENRICHMENT_BLOCKED`


In [15]:
# ============================================================
# 1.12 Сохранение итогового артефакта Stage 17
#
# Финальный actual state:
# - temporal enrichment текущего датасета заблокирован;
# - временной якорь восстановлен не будет;
# - новые модели не обучаются;
# - final test не используется.
# ============================================================

import json
from pathlib import Path
from datetime import datetime, timezone


ARTIFACT_PATH = Path(
    "../reports/generated/stage17_information_gap_map_V1.json"
)


print("▶ Формирование итогового артефакта Stage 17...", flush=True)


stage17_artifact = {
    "stage": "Stage 17",
    "version": "V1",

    "research_question": (
        "Какой информации потенциально не хватает текущему набору "
        "из 47 разрешённых признаков и можно ли корректно проверить "
        "дополнительные информационные гипотезы?"
    ),

    "status": "CURRENT_DATASET_TEMPORAL_ENRICHMENT_BLOCKED",

    "experiment_flags": {
        "model_training": False,
        "final_test_used": False,
        "quality_metrics_calculated": False,
        "new_predictors_added": False,
        "historical_enrichment_performed": False,
    },

    "feature_contract": {
        "allowed_feature_count": len(feature_names),
        "allowed_features": feature_names,
        "Q_B1_norm_used_as_predictor": False,
        "Q_B2_norm_used_as_predictor": False,
    },

    "business_semantics": {
        "confirmed_meaning_count": len(feature_meanings),
        "unknown_meaning_count": len(unknown_meanings),
        "confirmed_meanings": feature_meanings,
        "unknown_features": unknown_meanings,
    },

    "confirmed_information_domains": (
        domain_table.to_dict(orient="records")
    ),

    "information_coverage": (
        coverage_table.to_dict(orient="records")
    ),

    "customer_default_factors": (
        default_factor_table
        .astype(object)
        .where(default_factor_table.notna(), None)
        .to_dict(orient="records")
    ),

    "temporal_admissibility_gate": (
        temporal_gate.to_dict(orient="records")
    ),

    "future_dataset_contract": (
        future_dataset_contract.to_dict(orient="records")
    ),

    "data_source_map": (
        data_source_map.to_dict(orient="records")
    ),

    "confirmed_temporal_limits": (
        confirmed_temporal_limits.to_dict(orient="records")
    ),

    "key_counts": {
        "allowed_features": len(feature_names),
        "confirmed_feature_meanings": len(feature_meanings),
        "unknown_feature_meanings": len(unknown_meanings),

        "confirmed_information_domains": int(
            domain_table["Информационная область"].nunique()
        ),

        "customer_default_factors": len(default_factor_table),

        "direct_D1_D8_matches_confirmed": int(direct_matches),

        "related_D1_D8_areas_only": int(related_only),

        "additional_sources_considered": len(temporal_gate),

        "additional_sources_blocked_without_time_anchor": int(
            blocked_now
        ),

        "confirmed_temporal_blockers": len(
            confirmed_temporal_limits
        ),
    },

    "facts": [
        "Рабочий feature contract содержит 47 разрешённых признаков.",
        "Подтверждённый бизнес-смысл имеется для 17 признаков.",
        "Бизнес-смысл 30 признаков остаётся неподтверждённым.",
        "Прямое соответствие D_1-D_8 среди известных predictors не подтверждено.",
        "Для D_8 обнаружена только связанная область через Q_B3_norm.",
        "Все 10 рассмотренных дополнительных источников требуют временной привязки.",
        "Надёжная row-level observation_date или decision_date отсутствует.",
        "Заказчик подтвердил, что временной якорь для текущего датасета восстановлен не будет.",
        "Точная временная схема формирования DefMark относительно каждой строки также восстановлена не будет.",
    ],

    "interpretation": (
        "Существуют обоснованные гипотезы дополнительной информации, "
        "но их predictive value нельзя корректно проверить на текущем "
        "историческом датасете из-за отсутствия восстанавливаемой "
        "временной структуры."
    ),

    "limitations": [
        "Stage 17 не доказывает недостаточность текущих 47 признаков.",
        "Stage 17 не доказывает predictive value новых факторов.",
        "Смысл 30 разрешённых признаков остаётся неизвестным.",
        "Stage 17 не доказывает причинную природу blind spot.",
        "Random CV/OOF не доказывает temporal stability.",
        "Современные внешние snapshots нельзя использовать как исторические predictors.",
    ],

    "decision": {
        "current_dataset": (
            "Сохранить как воспроизводимый baseline для завершённого "
            "исследования моделей и blind spot."
        ),

        "historical_enrichment": (
            "Заблокирован по методологическим причинам."
        ),

        "model_only_search": (
            "Не возобновлять без новой гипотезы или нового источника информации."
        ),

        "next_research_object": (
            "Новый временно корректный датасет с row-level временным якорем."
        ),
    },

    "created_at_utc": datetime.now(timezone.utc).isoformat(),
}


ARTIFACT_PATH.parent.mkdir(
    parents=True,
    exist_ok=True
)


with ARTIFACT_PATH.open(
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        stage17_artifact,
        file,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    )


print("✅ Итоговый артефакт сохранён:")
print(ARTIFACT_PATH.resolve())

print()
print("Контроль:")
print(
    "status:",
    stage17_artifact["status"]
)

print(
    "allowed_features:",
    stage17_artifact["key_counts"]["allowed_features"]
)

print(
    "confirmed / unknown:",
    stage17_artifact["key_counts"]["confirmed_feature_meanings"],
    "/",
    stage17_artifact["key_counts"]["unknown_feature_meanings"],
)

print(
    "temporal blocked:",
    stage17_artifact["key_counts"][
        "additional_sources_blocked_without_time_anchor"
    ],
    "/",
    stage17_artifact["key_counts"]["additional_sources_considered"],
)

print(
    "confirmed temporal blockers:",
    stage17_artifact["key_counts"]["confirmed_temporal_blockers"]
)

print(
    "final_test_used:",
    stage17_artifact["experiment_flags"]["final_test_used"]
)

print(
    "model_training:",
    stage17_artifact["experiment_flags"]["model_training"]
)

▶ Формирование итогового артефакта Stage 17...
✅ Итоговый артефакт сохранён:
D:\Projects\komus-work\reports\generated\stage17_information_gap_map_V1.json

Контроль:
status: CURRENT_DATASET_TEMPORAL_ENRICHMENT_BLOCKED
allowed_features: 47
confirmed / unknown: 17 / 30
temporal blocked: 10 / 10
confirmed temporal blockers: 2
final_test_used: False
model_training: False


# 1.13 Что мы выяснили в Stage 17

## Факты

Текущий рабочий feature contract содержит 47 разрешённых признаков.

Для 17 признаков подтверждён бизнес-смысл.
Смысл ещё 30 признаков остаётся неподтверждённым.

Среди известных признаков уже представлены:

- финансовое состояние;
- размер бизнеса;
- отрасль и регион;
- кредитная история;
- исполнительные производства;
- отдельные организационные характеристики;
- характеристики руководства;
- отдельные показатели динамики бизнеса.

При этом в бизнес-методике заказчика существуют дополнительные события,
связанные с дефолтом: банкротство, ликвидация, отдельные исполнительные
и внутренние события Комуса.

Их прямое соответствие среди 17 расшифрованных predictors не подтверждено.

---

## Подтверждённое временное ограничение

Датасет содержит исторические наблюдения и `DefMark` за период 2005–2024,
но надёжной row-level даты наблюдения или даты принятия решения нет.

Заказчик подтвердил, что восстановить такой временной якорь для текущего
датасета невозможно.

Также отсутствует возможность восстановить точное правило формирования
`DefMark` относительно момента наблюдения для каждой исторической строки.

Следовательно, это ограничение больше не является открытым вопросом
или запросом к заказчику.

Оно является **подтверждённым ограничением текущего датасета**.

---

## Интерпретация

Stage 17 выявил обоснованные гипотезы дополнительной информации,
которая потенциально могла бы быть полезна для оценки риска.

Однако проверить эти гипотезы на текущем историческом датасете
методологически корректно невозможно.

Современные данные Spark, ФНС, судебных систем, ЕГРЮЛ и других источников
нельзя присоединять к историческим строкам только по `INN`,
поскольку невозможно доказать, что эти данные были доступны
на момент соответствующего исторического решения.

Такой эксперимент мог бы использовать будущую информацию
и создавать leakage.

---

## Что Stage 17 не доказывает

Stage 17 не доказывает:

- что текущие 47 признаков исчерпали всю информацию;
- что дополнительный источник обязательно улучшит Gini, Recall или PR-AUC;
- что blind spot вызван именно недостатком информации;
- что среди 30 нерасшифрованных признаков нет близких сигналов;
- что достигнут математический потолок качества модели.

---

## Исследовательское решение

Историческое внешнее enrichment текущего `Data_final.xlsb`
считается заблокированным из-за отсутствия временного якоря.

Продолжать:

- поиск новых моделей на неизменных 47 признаках;
- приклеивание современных внешних snapshots;
- подбор новых внешних факторов на текущих исторических строках

нерационально.

Следующий содержательный этап должен работать уже не с попыткой
доработать текущий historical dataset, а с вопросом:

**каким должен быть новый временно корректный датасет,
на котором можно честно исследовать современные источники
и новые факторы риска?**

---

## Финальный статус Stage 17

**CURRENT_DATASET_TEMPORAL_ENRICHMENT_BLOCKED**

Текущий датасет остаётся воспроизводимым baseline
для завершённого исследования моделей и blind spot.

Для нового исследования дополнительных источников требуется
другой датасет с корректной временной структурой.